In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Notebook parameters received")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}' (empty value defaults to FULL)")

In [0]:
source_table = f"adb_medsync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading data from Bronze")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent SQL query:")
print(f"    SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df=spark.read.table(source_table).filter(filter_condition)

bronze_record_count = bronze_df.count()
print(f"  Columns in source ({len(bronze_df.columns)}): {bronze_df.columns}")
print(f"  Records read from Bronze: {bronze_record_count}")
if bronze_record_count == 0:
    print("  WARNING: No records found for this landing_timestamp. Check the parameter value.")

In [0]:
from pyspark.sql import functions as F

target_table = f"adb_medsync.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print("=" * 80)
print("STEP 3: Validating load configuration")
print("=" * 80)
print(f"  Target table         : {target_table}")
print(f"  Raw load_type        : '{load_type}'")
print(f"  Normalized load_type : {normalized_load_type}")
print(f"  Parsed merge keys    : {merge_key_list}")


if normalized_load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")
    print(f"  Merge key validation passed: all keys {merge_key_list} exist in source data")
else:
    print(f"  Merge key validation skipped (not required for load_type {normalized_load_type})")

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp",F.current_timestamp())

print("  Added audit columns: insert_timestamp, update_timestamp (current_timestamp())")
print(f"  Final columns to load ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
import time

print("=" * 80)
print(f"STEP 4: Loading data into Silver using load_type = {normalized_load_type}")
print("=" * 80)
load_start = time.time()

if normalized_load_type == "FULL":
    print(f"  FULL load: overwriting table {target_table}")
    print("  Equivalent SQL query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp FROM adb_medsync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"  FULL load completed for {target_table}")
elif normalized_load_type == "APPEND":
    print(f"  APPEND load: appending rows to table {target_table}")
    print("  Equivalent SQL query:")
    print(f"    INSERT INTO {target_table} SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp FROM adb_medsync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"  APPEND load completed for {target_table}")
elif normalized_load_type == "MERGE":
    target_exists = spark.catalog.tableExists(target_table)
    print(f"  Target table exists: {target_exists}")

    if not target_exists:
        print(f"  Target table not found. Performing initial load (overwrite) into {target_table}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  Initial load completed for {target_table}")
    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"  Registered source data as temp view: {source_view}")

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """.strip()

        print(f"  Merge condition    : {merge_condition}")
        print(f"  Columns to update  : {[c for c in updated_df.columns if c != 'insert_timestamp']}")
        print(f"  Columns to insert  : {updated_df.columns}")
        print("  MERGE SQL query being executed:")
        print("-" * 80)
        print(merge_sql)
        print("-" * 80)

        merge_result = spark.sql(merge_sql)
        print("  MERGE completed. Merge metrics:")
        for row in merge_result.collect():
            for metric, value in row.asDict().items():
                print(f"    {metric}: {value}")
else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

print(f"  Load step finished in {time.time() - load_start:.2f} seconds")

In [0]:
record_count = updated_df.count()

print("=" * 80)
print("STEP 5: Finishing notebook")
print("=" * 80)
print(f"  Records processed for {table_name} (landing_timestamp = {landing_timestamp}): {record_count}")
print(f"  Exiting notebook with value: {record_count}")
dbutils.notebook.exit(str(record_count))